# 05 — Early detection analysis (Phase C_early)

Lead time of `PHT_recon` (reconstruction error) over `PHT_class` (classification error),
anticipated false positives and missed anticipations.

**Honest reporting**: if lead time is not significant or anticipated-FP rate is high, that is the result.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from driftgan.config import RESULTS_DIR

reports = {}
for path in sorted(Path(RESULTS_DIR).glob("runs/C_early/seed_*/lead_time.json")):
    seed = int(path.parent.name.split("_")[1])
    reports[seed] = json.loads(path.read_text())
pd.DataFrame([
    {"seed": seed,
     "mean_lead_time": r["mean_lead_time"],
     "n_pairs": len(r["pairs"]),
     "anticipated_fp_rate": r["anticipated_fp_rate"],
     "missed_anticipation_rate": r["missed_anticipation_rate"],
     "n_recon_fires": r["n_recon_fires"],
     "n_class_fires": r["n_class_fires"]}
    for seed, r in reports.items()
])

In [ ]:
# Lead-time distribution across seeds
import matplotlib.pyplot as plt

leads = [lt for r in reports.values() for lt in r["lead_times"]]
if leads:
    plt.hist(leads, bins=30, color="#2c3e50", alpha=0.8)
    plt.xlabel("lead time (samples)")
    plt.ylabel("paired events")
    plt.title("PHT_recon anticipation over PHT_class")
else:
    print("no paired fires — no evidence of anticipation")

In [ ]:
# Compare C_early recovery vs B and C (B/C under shared schedule, C_early dynamic)
from scripts.make_report import load_recovery_frames

from driftgan.viz.plots import recovery_boxplots

recovery = load_recovery_frames(Path(RESULTS_DIR))
subset = {p: df for p, df in recovery.items() if p in ("B", "C", "C_early")}
if subset:
    recovery_boxplots(subset, "AUD")